In [4]:
import pandas as pd 
from pathlib import Path

#Import "pandas" could not be resolved from source


#Load the customer reviews dataset

file_path = Path("/Users/poojanair/Downloads/UrbanKart_HuggingFace_NLP_Customer_Reviews.csv")

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

print("\nColumn names:")

print(df.columns.tolist())

Dataset shape: (6000, 16)

Column names:
['Customer_ID', 'review_text', 'sentiment_label', 'satisfaction_score', 'preferred_category', 'preferred_channel', 'customer_value_segment', 'customer_tenure_months', 'orders_last_12m', 'days_since_last_purchase', 'support_tickets_12m', 'return_rate', 'churn', 'purchase_next_30_days', 'data_origin', 'split']


In [5]:
print("\nFirst five reviews:")
display(df[["review_text", "sentiment_label", "split"]])


First five reviews:


,review_text,sentiment_label,split
0,I am unhappy with my Sports & Fitness order an...,negative,train
1,I was disappointed with my Electronics purchas...,negative,test
2,I was disappointed with my Sports & Fitness pu...,negative,train
3,The Books shopping experience needs improvemen...,negative,train
4,"My Sports & Fitness order was frustrating, and...",negative,validation
...,...,...,...
5995,I liked the Accessories selection and had a po...,positive,validation
5996,I had a great experience shopping for Beauty &...,positive,train
5997,"The Electronics order arrived as expected, and...",positive,train
5998,Good value for money on my Sports & Fitness or...,positive,train


In [6]:
print("\nSentiment distribution:")
print(df["sentiment_label"].value_counts())


Sentiment distribution:
sentiment_label
positive    3287
neutral     1988
negative     725
Name: count, dtype: int64


In [7]:
print("\nMissing review text:", df["review_text"].isna().sum())
print("Missing sentiment labels:", df["sentiment_label"].isna().sum())


Missing review text: 0
Missing sentiment labels: 0


In [8]:
! pip install transformers

In [9]:
%pip install torch
import torch

print("PyTorch version:", torch.__version__)
print("PyTorch installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 16.1 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 20.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 7.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [torch]32m3/4 [torch]kx]
Note: you may need to restart the kernel to use updated packages.
PyTorch version: 2.14.1
PyTorch installed successfully!


In [10]:
import sys

print("Python executable:", sys.executable)

import torch

print("PyTorch version:", torch.__version__)
print("PyTorch location:", torch.__file__)

Python executable: /opt/anaconda3/envs/urbanskart_regression/bin/python
PyTorch version: 2.14.1
PyTorch location: /opt/anaconda3/envs/urbanskart_regression/lib/python3.11/site-packages/torch/__init__.py


In [11]:
import torch
import transformers
from transformers import pipeline

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Transformers location:", transformers.__file__)

/opt/anaconda3/envs/urbanskart_regression/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.14.1
Transformers: 5.19.0
Transformers location: /opt/anaconda3/envs/urbanskart_regression/lib/python3.11/site-packages/transformers/__init__.py


In [12]:
from transformers import pipeline  
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME
)
print("Sentiment model loaded successfully.")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 48858.60it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model loaded successfully.


In [13]:
test_reviews = [


"The product quality is excellent and delivery was fast.",

"The product arrived damaged and customer support was unhelpful.",

"The package arrived yesterday."
]

for review in test_reviews:
    result = sentiment_analyzer(review) [0]
    print("Review:", review)
    print("Sentiment:", result["label"])
    print("Score:", round (result["score"], 4))
    print("-"* 60)

Review: The product quality is excellent and delivery was fast.
Sentiment: positive
Score: 0.9794
------------------------------------------------------------
Review: The product arrived damaged and customer support was unhelpful.
Sentiment: negative
Score: 0.9395
------------------------------------------------------------
Review: The package arrived yesterday.
Sentiment: neutral
Score: 0.5292
------------------------------------------------------------


In [14]:
print("Dataset split distribution:")
print(df["split"].value_counts())

print("\nSentiment labels:")
print(df["sentiment_label"].value_counts())

Dataset split distribution:
split
train         4800
test           600
validation     600
Name: count, dtype: int64

Sentiment labels:
sentiment_label
positive    3287
neutral     1988
negative     725
Name: count, dtype: int64


In [15]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Select held-out test data only
test_df = df.loc[
    df["split"].str.lower() == "test",
    ["review_text", "sentiment_label"]
].dropna().copy()

print("Test records:", len(test_df))

# Predict in batches
predictions = sentiment_analyzer(
    test_df["review_text"].astype(str).tolist(),
    batch_size=16,
    truncation=True
)

Test records: 600


In [16]:
df.head()

,Customer_ID,review_text,sentiment_label,satisfaction_score,preferred_category,preferred_channel,customer_value_segment,customer_tenure_months,orders_last_12m,days_since_last_purchase,support_tickets_12m,return_rate,churn,purchase_next_30_days,data_origin,split
0,CUST009609,I am unhappy with my Sports & Fitness order an...,negative,2.00,Sports & Fitness,Website,Low Value,26,8,84,4,0.190,1,0,synthetic_generated_for_training,train
1,CUST004601,I was disappointed with my Electronics purchas...,negative,1.56,Electronics,Mobile App,Medium Value,10,7,63,4,0.000,1,1,synthetic_generated_for_training,test
2,CUST000722,I was disappointed with my Sports & Fitness pu...,negative,1.89,Sports & Fitness,Website,Low Value,12,6,73,6,0.119,1,0,synthetic_generated_for_training,train
3,CUST007179,The Books shopping experience needs improvemen...,negative,2.41,Books,Physical Store,Low Value,12,4,322,0,0.210,1,0,synthetic_generated_for_training,train
4,CUST002160,"My Sports & Fitness order was frustrating, and...",negative,2.46,Sports & Fitness,Marketplace,Medium Value,20,6,36,2,0.327,0,1,synthetic_generated_for_training,validation
